## Library Imports and Configs

In [60]:
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
import numpy as np
import pandas as pd
pd.set_option('display.max_columns', None)
import os
import sklearn
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import OrdinalEncoder
import optuna
import xgboost as xgb 
import lightgbm as lgb
import catboost as cb
from optbinning import OptimalBinning
from sklearn.model_selection import StratifiedKFold, train_test_split
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score
import plotly.graph_objects as go
import math
import random
import warnings
from tqdm import tqdm
import numpy as np, pandas as pd
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.utils.class_weight import compute_class_weight
from sklearn.preprocessing import KBinsDiscretizer, TargetEncoder
import os
import torch
import torch.nn as nn
import torch.nn.functional as F

warnings.filterwarnings('ignore')
print("PyTorch  version:", torch.__version__)

def seed_everything(seed: int):
    np.random.seed(seed)
    random.seed(seed)
    torch.manual_seed(seed)
seed_everything(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

PyTorch  version: 2.11.0+cpu


In [31]:
ROOT_PATH = "playground-series-s6e5"

train_df = pd.read_csv(os.path.join(ROOT_PATH, "train.csv"))
test_df = pd.read_csv(os.path.join(ROOT_PATH, "test.csv"))
sub_df = pd.read_csv(os.path.join(ROOT_PATH, "sample_submission.csv"))

train_df.shape, test_df.shape 

((439140, 16), (188165, 15))

In [32]:
F_ENGG = True
MAX_BIN = 25
MIN_BIN = 0.06
SHOW_WOE = False
OPTUNA = True

## Feature Engineering
- Domain Knowledge
- Correlation after feature engineering

In [33]:
def engineer_race_features(df, activate=True):

    if not activate:
        return df

    df = df.copy()

    eps = 1e-5

    # =========================================================
    # BASIC PROGRESS FEATURES
    # =========================================================

    df['Progress_Per_Lap_engg'] = (
        df['RaceProgress'] / (df['LapNumber'] + eps)
    )

    df['Remaining_RaceProgress_engg'] = (
        1 - df['RaceProgress']
    )

    df['Remaining_Laps_Ratio_engg'] = (
        (1 - df['RaceProgress']) /
        (df['LapNumber'] + eps)
    )

    # =========================================================
    # DEGRADATION FEATURES
    # =========================================================

    df['Deg_Per_Lap_engg'] = (
        df['Cumulative_Degradation'] /
        (df['LapNumber'] + eps)
    )

    df['Deg_Per_TyreLife_engg'] = (
        df['Cumulative_Degradation'] /
        (df['TyreLife'] + eps)
    )

    df['Deg_x_TyreLife_engg'] = (
        df['Cumulative_Degradation'] *
        df['TyreLife']
    )

    df['Deg_x_Progress_engg'] = (
        df['Cumulative_Degradation'] *
        df['RaceProgress']
    )

    df['Deg_Acceleration_engg'] = (
        df['Cumulative_Degradation'] /
        (df['RaceProgress'] + eps)
    )

    # =========================================================
    # PACE FEATURES
    # =========================================================

    df['Pace_Tyre_Sensitivity_engg'] = (
        df['LapTime_Delta'] /
        (df['TyreLife'] + eps)
    )

    df['Pace_Per_Position_engg'] = (
        df['LapTime_Delta'] /
        (df['Position'] + eps)
    )

    df['LapTime_x_TyreLife_engg'] = (
        df['LapTime_Delta'] *
        df['TyreLife']
    )

    df['LapTime_x_Deg_engg'] = (
        df['LapTime_Delta'] *
        df['Cumulative_Degradation']
    )

    df['LapTime_x_Progress_engg'] = (
        df['LapTime_Delta'] *
        df['RaceProgress']
    )

    df['Pace_Drop_Flag_engg'] = (
        df['LapTime_Delta'] > 0
    ).astype(int)

    df['Extreme_Pace_Drop_engg'] = (
        df['LapTime_Delta'] > df['LapTime_Delta'].quantile(0.90)
    ).astype(int)

    # =========================================================
    # TYRE FEATURES
    # =========================================================

    df['TyreLife_Per_Lap_engg'] = (
        df['TyreLife'] /
        (df['LapNumber'] + eps)
    )

    df['TyreLife_Per_Progress_engg'] = (
        df['TyreLife'] /
        (df['RaceProgress'] + eps)
    )

    df['TyreLife_x_Progress_engg'] = (
        df['TyreLife'] *
        df['RaceProgress']
    )

    df['Fresh_Tyre_Flag_engg'] = (
        df['TyreLife'] <= 5
    ).astype(int)

    df['Medium_Tyre_Flag_engg'] = (
        (df['TyreLife'] > 5) &
        (df['TyreLife'] <= 20)
    ).astype(int)

    df['Old_Tyre_Flag_engg'] = (
        df['TyreLife'] > 20
    ).astype(int)

    # =========================================================
    # POSITION FEATURES
    # =========================================================

    df['Losing_Ground_engg'] = (
        df['Position_Change'] < 0
    ).astype(int)

    df['Gaining_Ground_engg'] = (
        df['Position_Change'] > 0
    ).astype(int)

    df['Position_x_Progress_engg'] = (
        df['Position'] *
        df['RaceProgress']
    )

    df['Position_x_TyreLife_engg'] = (
        df['Position'] *
        df['TyreLife']
    )

    df['Position_Change_Intensity_engg'] = (
        df['Position_Change'] /
        (df['LapNumber'] + eps)
    )

    df['Bad_Position_Flag_engg'] = (
        df['Position'] > 10
    ).astype(int)

    df['Podium_Position_Flag_engg'] = (
        df['Position'] <= 3
    ).astype(int)

    # =========================================================
    # PIT WINDOW FEATURES
    # =========================================================

    df['Potential_Pit_Window_engg'] = (
        (df['TyreLife'] > 15) &
        (df['RaceProgress'] > 0.25)
    ).astype(int)

    df['Late_Race_Pit_Window_engg'] = (
        (df['RaceProgress'] > 0.70)
    ).astype(int)

    df['Early_Race_Pit_Window_engg'] = (
        (df['RaceProgress'] < 0.25)
    ).astype(int)

    # =========================================================
    # INTERACTION FEATURES
    # =========================================================

    df['Wear_Pace_Impact_engg'] = (
        df['LapTime_Delta'] *
        df['Cumulative_Degradation']
    )

    df['Wear_Position_Impact_engg'] = (
        df['Cumulative_Degradation'] *
        df['Position']
    )

    df['Wear_Position_Change_Impact_engg'] = (
        df['Cumulative_Degradation'] *
        df['Position_Change']
    )

    df['TyreLife_Position_Interaction_engg'] = (
        df['TyreLife'] *
        df['Position']
    )

    df['TyreLife_Lap_Interaction_engg'] = (
        df['TyreLife'] *
        df['LapNumber']
    )

    df['TyreLife_Stint_Interaction_engg'] = (
        df['TyreLife'] *
        df['Stint']
    )

    df['Lap_Position_Interaction_engg'] = (
        df['LapNumber'] *
        df['Position']
    )

    # =========================================================
    # STINT FEATURES
    # =========================================================

    df['Is_First_Stint_engg'] = (
        df['Stint'] == 1
    ).astype(int)

    df['Is_Second_Stint_engg'] = (
        df['Stint'] == 2
    ).astype(int)

    df['Is_ThirdPlus_Stint_engg'] = (
        df['Stint'] >= 3
    ).astype(int)

    df['Stint_x_Progress_engg'] = (
        df['Stint'] *
        df['RaceProgress']
    )

    df['Stint_x_TyreLife_engg'] = (
        df['Stint'] *
        df['TyreLife']
    )

    # =========================================================
    # CATEGORICAL COMBINATIONS
    # =========================================================

    df['Year_Stint_engg'] = (
        df['Year'].astype(str)
        + "_"
        + df['Stint'].astype(str)
    )

    df['Compound_Year_engg'] = (
        df['Compound'].astype(str)
        + "_"
        + df['Year'].astype(str)
    )

    df['Race_Stint_engg'] = (
        df['Race'].astype(str)
        + "_"
        + df['Stint'].astype(str)
    )

    df['Driver_Compound_engg'] = (
        df['Driver'].astype(str)
        + "_"
        + df['Compound'].astype(str)
    )

    df['Driver_Race_engg'] = (
        df['Driver'].astype(str)
        + "_"
        + df['Race'].astype(str)
    )

    df['Compound_Stint_engg'] = (
        df['Compound'].astype(str)
        + "_"
        + df['Stint'].astype(str)
    )

    df['Compound_Position_engg'] = (
        df['Compound'].astype(str)
        + "_"
        + df['Position'].astype(str)
    )

    df['Race_Compound_engg'] = (
        df['Race'].astype(str)
        + "_"
        + df['Compound'].astype(str)
    )

    df['Race_Year_engg'] = (
        df['Race'].astype(str)
        + "_"
        + df['Year'].astype(str)
    )

    df['Driver_Stint_engg'] = (
        df['Driver'].astype(str)
        + "_"
        + df['Stint'].astype(str)
    )

    return df

print(f"Applying feature engineering on train and test ...")
train_df_eng = engineer_race_features(train_df)
test_df_eng = engineer_race_features(test_df)

print(f"Seperating categorical and numerical columns ...")
c_cols = test_df_eng.drop(['id'], axis='columns').select_dtypes(include='object').columns
n_cols = test_df_eng.drop(['id'], axis='columns').select_dtypes(exclude='object').columns
t_col = "PitNextLap"
ID = "id"

Applying feature engineering on train and test ...
Seperating categorical and numerical columns ...


C:\Users\admin\AppData\Local\Temp\ipykernel_14736\3101169399.py:313: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  c_cols = test_df_eng.drop(['id'], axis='columns').select_dtypes(include='object').columns


## Data Processing
- Encoding - (categorical data) (Label Encoding and OneHotEncoding)
- Normalization - (numerical data)

#### Encoding - Categorical data - Label Encoding and OneHotEncoding

In [34]:
encoding_columns = ['Driver', 'Race', 'Year_Stint_engg', 'Compound_Year_engg',
       'Race_Stint_engg', 'Driver_Compound_engg', 'Driver_Race_engg',
       'Compound_Stint_engg', 'Compound_Position_engg', 'Race_Compound_engg',
       'Race_Year_engg', 'Driver_Stint_engg']

ord_encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
one_hot_encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')

print(f"Applying Ordinal Encoding ...")
train_df_eng[[col.replace("engg", "en") if "engg" in col  else col + "_en" for col in encoding_columns]] = ord_encoder.fit_transform(train_df_eng[encoding_columns])
test_df_eng[[col.replace("engg", "en") if "engg" in col  else col + "_en" for col in encoding_columns]] = ord_encoder.transform(test_df_eng[encoding_columns])

train_df_eng_enc = train_df_eng.drop(encoding_columns, axis='columns')
test_df_eng_enc = test_df_eng.drop(encoding_columns, axis='columns')


print(f"Applying One Hot Encoding ...")
encoded_data = one_hot_encoder.fit_transform(train_df_eng_enc[['Compound']])
test_encoded_data = one_hot_encoder.transform(test_df_eng_enc[['Compound']])

train_encoded_df = pd.DataFrame(encoded_data, columns=one_hot_encoder.get_feature_names_out(['Compound']), index=train_df_eng_enc.index)
test_encoded_df = pd.DataFrame(test_encoded_data, columns=one_hot_encoder.get_feature_names_out(['Compound']), index=test_df_eng_enc.index)

train_df_eng_enc_v1 = pd.concat([train_df_eng_enc.drop('Compound', axis=1), train_encoded_df], axis=1)
test_df_eng_enc_v1 = pd.concat([test_df_eng_enc.drop('Compound', axis=1), test_encoded_df], axis=1)

test_df_eng_enc_v1.head()

Applying Ordinal Encoding ...
Applying One Hot Encoding ...


,id,Year,PitStop,LapNumber,Stint,TyreLife,Position,LapTime (s),LapTime_Delta,Cumulative_Degradation,RaceProgress,Position_Change,Progress_Per_Lap_engg,Remaining_RaceProgress_engg,Remaining_Laps_Ratio_engg,Deg_Per_Lap_engg,Deg_Per_TyreLife_engg,Deg_x_TyreLife_engg,Deg_x_Progress_engg,Deg_Acceleration_engg,Pace_Tyre_Sensitivity_engg,Pace_Per_Position_engg,LapTime_x_TyreLife_engg,LapTime_x_Deg_engg,LapTime_x_Progress_engg,Pace_Drop_Flag_engg,Extreme_Pace_Drop_engg,TyreLife_Per_Lap_engg,TyreLife_Per_Progress_engg,TyreLife_x_Progress_engg,Fresh_Tyre_Flag_engg,Medium_Tyre_Flag_engg,Old_Tyre_Flag_engg,Losing_Ground_engg,Gaining_Ground_engg,Position_x_Progress_engg,Position_x_TyreLife_engg,Position_Change_Intensity_engg,Bad_Position_Flag_engg,Podium_Position_Flag_engg,Potential_Pit_Window_engg,Late_Race_Pit_Window_engg,Early_Race_Pit_Window_engg,Wear_Pace_Impact_engg,Wear_Position_Impact_engg,Wear_Position_Change_Impact_engg,TyreLife_Position_Interaction_engg,TyreLife_Lap_Interaction_engg,TyreLife_Stint_Interaction_engg,Lap_Position_Interaction_engg,Is_First_Stint_engg,Is_Second_Stint_engg,Is_ThirdPlus_Stint_engg,Stint_x_Progress_engg,Stint_x_TyreLife_engg,Driver_en,Race_en,Year_Stint_en,Compound_Year_en,Race_Stint_en,Driver_Compound_en,Driver_Race_en,Compound_Stint_en,Compound_Position_en,Race_Compound_en,Race_Year_en,Driver_Stint_en,Compound_HARD,Compound_INTERMEDIATE,Compound_MEDIUM,Compound_SOFT,Compound_WET
0,439140,2023,0,21,1,21.0,4,93.387,0.280,-4.984,0.403846,0.0,0.019231,0.596154,0.028388,-0.237333,-0.237333,-104.664,-2.012769,-12.341028,0.013333,0.070000,5.880,-1.395520,0.113077,1,0,1.000000,51.998712,8.480769,0,0,1,0,0,1.615385,84.0,0.000000,0,0,1,0,0,-1.395520,-19.936,-0.000,84.0,441.0,21.0,84,1,0,0,0.403846,21.0,144.0,6.0,8.0,9.0,36.0,704.0,3747.0,15.0,54.0,25.0,25.0,895.0,0.0,0.0,1.0,0.0,0.0
1,439141,2023,0,24,1,24.0,1,90.867,-0.129,-1.990,0.413793,0.0,0.017241,0.586207,0.024425,-0.082917,-0.082917,-47.760,-0.823448,-4.809050,-0.005375,-0.128999,-3.096,0.256710,-0.053379,0,0,1.000000,57.998598,9.931034,0,0,1,0,0,0.413793,24.0,0.000000,0,1,1,0,0,0.256710,-1.990,-0.000,24.0,576.0,24.0,24,1,0,0,0.413793,24.0,875.0,0.0,8.0,9.0,0.0,2890.0,14631.0,15.0,40.0,2.0,1.0,3777.0,0.0,0.0,1.0,0.0,0.0
2,439142,2023,0,24,1,24.0,11,92.871,0.041,-8.842,0.461538,0.0,0.019231,0.538462,0.022436,-0.368417,-0.368417,-212.208,-4.080923,-19.157252,0.001708,0.003727,0.984,-0.362522,0.018923,1,0,1.000000,51.998873,11.076923,0,0,1,0,0,5.076923,264.0,0.000000,1,0,1,0,0,-0.362522,-97.262,-0.000,264.0,576.0,24.0,264,1,0,0,0.461538,24.0,295.0,6.0,8.0,9.0,36.0,1276.0,7495.0,15.0,42.0,25.0,25.0,1704.0,0.0,0.0,1.0,0.0,0.0
3,439143,2024,0,6,2,4.0,15,94.967,-19.741,8.250,0.077922,1.0,0.012987,0.922078,0.153679,1.374998,2.062495,33.000,0.642857,105.861414,-4.935238,-1.316066,-78.964,-162.863250,-1.538260,0,0,0.666666,51.326746,0.311688,1,0,0,0,1,1.168831,60.0,0.166666,1,0,0,0,1,-162.863250,123.750,8.250,60.0,24.0,8.0,90,0,1,0,0.155844,8.0,137.0,24.0,17.0,14.0,140.0,672.0,3583.0,24.0,66.0,93.0,98.0,856.0,0.0,0.0,0.0,1.0,0.0
4,439144,2024,0,52,2,29.0,12,99.112,0.930,-20.848,0.722222,7.0,0.013889,0.277778,0.005342,-0.400923,-0.718896,-604.592,-15.056889,-28.866062,0.032069,0.077500,26.970,-19.388640,0.671667,1,0,0.557692,40.153290,20.944444,0,0,1,0,1,8.666667,348.0,0.134615,1,0,1,1,0,-19.388640,-250.176,-145.936,348.0,1508.0,58.0,624,0,1,0,1.444444,58.0,4.0,25.0,17.0,2.0,147.0,20.0,129.0,1.0,3.0,95.0,102.0,28.0,1.0,0.0,0.0,0.0,0.0


#### Normalization (MinMax / StandardScaler)

In [35]:
from sklearn.preprocessing import MinMaxScaler, StandardScaler

class DataNormalizer:
    def __init__(self, method='standard'):
        """
        Initializes the normalizer with the chosen scaling strategy.
        method: 'minmax' or 'standard'
        """
        self.method = method
        if method == 'minmax':
            self.scaler = MinMaxScaler()
        elif method == 'standard':
            self.scaler = StandardScaler()
        else:
            raise ValueError("Method must be either 'minmax' or 'standard'")

    def fit(self, dataset, n_cols):
        """
        Learns the scaling parameters (mean/std or min/max) from the training set.
        """
        if not all(col in dataset.columns for col in n_cols):
            missing = [c for c in n_cols if c not in dataset.columns]
            raise ValueError(f"Columns missing from dataset: {missing}")
            
        self.scaler.fit(dataset[n_cols])
        print(f"Successfully fitted {self.method} scaler on: {n_cols}")

    def transform(self, dataset, n_cols):
        """
        Applies the learned parameters to scale the dataset.
        """
        df = dataset.copy()
        df[n_cols] = self.scaler.transform(df[n_cols])
        return df

    def fit_transform(self, dataset, n_cols):
        """
        Fits to the data then transforms it. Useful for the initial training set.
        """
        self.fit(dataset, n_cols)
        return self.transform(dataset, n_cols)


normalizer = DataNormalizer(method='standard')

train_df_scaled = normalizer.fit_transform(train_df_eng_enc_v1, n_cols=n_cols)
test_df_scaled = normalizer.transform(test_df_eng_enc_v1, n_cols=n_cols)

train_df_scaled.head()

Successfully fitted standard scaler on: Index(['Year', 'PitStop', 'LapNumber', 'Stint', 'TyreLife', 'Position',
       'LapTime (s)', 'LapTime_Delta', 'Cumulative_Degradation',
       'RaceProgress', 'Position_Change', 'Progress_Per_Lap_engg',
       'Remaining_RaceProgress_engg', 'Remaining_Laps_Ratio_engg',
       'Deg_Per_Lap_engg', 'Deg_Per_TyreLife_engg', 'Deg_x_TyreLife_engg',
       'Deg_x_Progress_engg', 'Deg_Acceleration_engg',
       'Pace_Tyre_Sensitivity_engg', 'Pace_Per_Position_engg',
       'LapTime_x_TyreLife_engg', 'LapTime_x_Deg_engg',
       'LapTime_x_Progress_engg', 'Pace_Drop_Flag_engg',
       'Extreme_Pace_Drop_engg', 'TyreLife_Per_Lap_engg',
       'TyreLife_Per_Progress_engg', 'TyreLife_x_Progress_engg',
       'Fresh_Tyre_Flag_engg', 'Medium_Tyre_Flag_engg', 'Old_Tyre_Flag_engg',
       'Losing_Ground_engg', 'Gaining_Ground_engg', 'Position_x_Progress_engg',
       'Position_x_TyreLife_engg', 'Position_Change_Intensity_engg',
       'Bad_Position_Flag_engg', 

,id,Year,PitStop,LapNumber,Stint,TyreLife,Position,LapTime (s),LapTime_Delta,Cumulative_Degradation,RaceProgress,Position_Change,PitNextLap,Progress_Per_Lap_engg,Remaining_RaceProgress_engg,Remaining_Laps_Ratio_engg,Deg_Per_Lap_engg,Deg_Per_TyreLife_engg,Deg_x_TyreLife_engg,Deg_x_Progress_engg,Deg_Acceleration_engg,Pace_Tyre_Sensitivity_engg,Pace_Per_Position_engg,LapTime_x_TyreLife_engg,LapTime_x_Deg_engg,LapTime_x_Progress_engg,Pace_Drop_Flag_engg,Extreme_Pace_Drop_engg,TyreLife_Per_Lap_engg,TyreLife_Per_Progress_engg,TyreLife_x_Progress_engg,Fresh_Tyre_Flag_engg,Medium_Tyre_Flag_engg,Old_Tyre_Flag_engg,Losing_Ground_engg,Gaining_Ground_engg,Position_x_Progress_engg,Position_x_TyreLife_engg,Position_Change_Intensity_engg,Bad_Position_Flag_engg,Podium_Position_Flag_engg,Potential_Pit_Window_engg,Late_Race_Pit_Window_engg,Early_Race_Pit_Window_engg,Wear_Pace_Impact_engg,Wear_Position_Impact_engg,Wear_Position_Change_Impact_engg,TyreLife_Position_Interaction_engg,TyreLife_Lap_Interaction_engg,TyreLife_Stint_Interaction_engg,Lap_Position_Interaction_engg,Is_First_Stint_engg,Is_Second_Stint_engg,Is_ThirdPlus_Stint_engg,Stint_x_Progress_engg,Stint_x_TyreLife_engg,Driver_en,Race_en,Year_Stint_en,Compound_Year_en,Race_Stint_en,Driver_Compound_en,Driver_Race_en,Compound_Stint_en,Compound_Position_en,Race_Compound_en,Race_Year_en,Driver_Stint_en,Compound_HARD,Compound_INTERMEDIATE,Compound_MEDIUM,Compound_SOFT,Compound_WET
0,0,-1.486487,-0.396946,1.585901,0.221941,2.534531,-0.308849,-0.630046,-0.086333,0.853455,1.487008,1.222548,1.0,-0.091033,-1.487008,-0.509986,0.151346,0.220112,1.196900,1.220405,0.142041,0.024664,-0.023204,-1.253837,-0.019963,-0.979879,-0.660523,-0.333333,-0.019388,-0.019540,2.812363,-0.509979,-1.130327,1.816110,-0.688951,1.320038,0.745101,1.515992,0.098721,-0.899887,-0.433810,1.403299,2.817278,-0.925252,-0.019963,0.747463,0.380384,1.515992,2.875356,2.040728,0.802183,-0.985163,1.545995,-0.519458,0.741509,2.040728,134.0,7.0,1.0,0.0,43.0,654.0,3488.0,1.0,18.0,27.0,28.0,838.0,1.0,0.0,0.0,0.0,0.0
1,1,1.440545,2.519236,0.229628,0.221941,-0.730333,-1.066602,-0.801797,-0.656423,-3.605949,0.033534,-0.774077,0.0,-0.401788,-0.033534,-0.416683,-0.384754,-1.788677,-1.095286,-3.176571,-0.405659,-0.088712,-0.585639,-0.928667,0.365547,-2.250847,-0.660523,-0.333333,-1.234165,-1.149458,-0.509855,-0.509979,0.884700,-0.550627,1.451482,-0.757554,-0.569919,-0.850941,-0.099932,-0.899887,-0.433810,-0.712606,-0.354953,-0.925252,0.365547,-1.124840,2.373826,-0.850941,-0.466697,-0.510165,-0.518896,-0.985163,1.545995,-0.519458,-0.093803,-0.510165,111.0,9.0,23.0,3.0,54.0,545.0,2892.0,1.0,14.0,34.0,39.0,700.0,1.0,0.0,0.0,0.0,0.0
2,2,-1.486487,-0.396946,2.116616,1.274359,0.800072,0.638343,-1.011682,-0.085787,-1.365930,1.902200,0.723392,1.0,-0.175196,-1.902200,-0.523369,0.020257,-0.096360,-1.719948,-3.420249,0.018482,0.020887,0.005312,-0.624164,0.027555,-1.147342,-0.660523,-0.333333,-0.969109,-0.931692,1.528443,-0.509979,-1.130327,1.816110,-0.688951,1.320038,2.245034,1.299301,0.052469,1.111250,-0.433810,1.403299,2.817278,-0.925252,0.027555,-1.855510,-1.055683,1.299301,1.637980,1.562435,2.462581,-0.985163,-0.646833,1.925083,1.909803,1.562435,886.0,2.0,2.0,0.0,14.0,2942.0,14918.0,2.0,4.0,8.0,8.0,3855.0,1.0,0.0,0.0,0.0,0.0
3,3,-0.510810,-0.396946,-1.244581,-0.830476,-1.240468,-0.498287,0.172574,-0.080872,0.335931,-1.029455,-0.025343,0.0,5.036382,1.029455,1.788622,-0.100579,-0.040139,0.394036,0.478171,0.040803,-0.063385,-0.031068,0.113365,-0.008944,0.064819,-0.660523,-0.333333,0.493814,-0.959647,-0.806264,1.960864,-1.130327,-0.550627,-0.688951,-0.757554,-0.826915,-0.967621,0.004622,-0.899887,-0.433810,-0.712606,-0.354953,1.080787,-0.008944,0.360254,0.009275,-0.967621,-0.817793,-0.988458,-0.944175,1.015061,-0.646833,-0.519458,-0.791974,-0.988458,864.0,19.0,8.0,9.0,111.0,2836.0,14364.0,15.0,57.0,72.0,77.0,3703.0,0.0,0.0,1.0,0.0,0.0
4,4,-1.486487,2.519236,0.170660,1.274359,-0.832360,-1.445478,0.856192,0.289790,0.211493,0.092589,0.723392,0.0,-0.175196,-0.

## Target Encoding - OOF

In [48]:
from sklearn.model_selection import StratifiedKFold

encoded_cat_cols = [col for col in train_df_scaled.columns if '_en' in col.lower() and '_engg' not in col.lower()]
encoded_cat_cols

train_df_scaled_v1 = train_df_scaled.copy()
test_df_scaled_v1 = test_df_scaled.copy()

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for col in encoded_cat_cols:
    train_df_scaled_v1[f'{col}_mean'] = 0.0
    train_df_scaled_v1[f'{col}_std'] = 0.0

X, y = train_df_scaled_v1.drop([t_col], axis='columns'), train_df_scaled_v1[t_col]

for train_idx, val_idx in skf.split(X, y):
    train_fold = train_df_scaled_v1.iloc[train_idx]
    val_fold = train_df_scaled_v1.iloc[val_idx]

    for col in encoded_cat_cols:
        stats = train_fold.groupby(col)[t_col].agg(['mean', 'std'])
        val_fold = val_fold.merge(stats, on=col, how='left')

        train_df_scaled_v1.loc[val_idx, f'{col}_mean'] = val_fold['mean'].values
        train_df_scaled_v1.loc[val_idx, f'{col}_std'] = val_fold['std'].values

        val_fold.drop(['mean', 'std'], axis=1, inplace=True)

for col in encoded_cat_cols:
    train_df_scaled_v1[[f'{col}_mean', f'{col}_std']] = train_df_scaled_v1[
        [f'{col}_mean', f'{col}_std']
    ].fillna(0)

for col in encoded_cat_cols:
    stats = train_df_scaled_v1.groupby(col)[t_col].agg(['mean', 'std'])
    test_df_scaled_v1 = test_df_scaled_v1.merge(stats, on=col, how='left')

    test_df_scaled_v1[f'{col}_mean'] = test_df_scaled_v1['mean']
    test_df_scaled_v1[f'{col}_std'] = test_df_scaled_v1['std']

    test_df_scaled_v1.drop(['mean', 'std'], axis=1, inplace=True)

for col in encoded_cat_cols:
    test_df_scaled_v1[[f'{col}_mean', f'{col}_std']] = test_df_scaled_v1[
        [f'{col}_mean', f'{col}_std']
    ].fillna(0)

## WoE Transformation
- WoE
- IV

In [93]:
class OptimalBinner:

    def __init__(
        self, c_cols: list, n_cols: list, max_n_bins: int = 5, min_bin_size: float = 0.05):
        self.c_cols = c_cols
        self.n_cols = n_cols
        self.max_n_bins = max_n_bins
        self.min_bin_size = min_bin_size
        self.binners = {}

    def fit(self, X, y, verbos=True):
        for col in X.columns:
            if verbos:
                print(f"Fitting: {col}")
            if col in self.c_cols:
                optb = OptimalBinning(
                    name=col,
                    dtype="categorical",
                    max_n_bins=self.max_n_bins,
                    min_bin_size=self.min_bin_size
                )

            else:
                optb = OptimalBinning(
                    name=col,
                    dtype="numerical",
                    max_n_bins=self.max_n_bins,
                    min_bin_size=self.min_bin_size
                )
            # Fit binning
            optb.fit(X[col], y)
            self.binners[col] = optb
        return self

    def transform(self, X, metric="woe"):
        X_transformed = pd.DataFrame(index=X.index)
        for col in X.columns:
            optb = self.binners[col]
            X_transformed[col] = optb.transform(
                X[col],
                metric=metric,
                metric_missing="empirical",
                metric_special="empirical"
            )
        return X_transformed

    def get_iv_summary(self):
        iv_data = []
        for col, optb in self.binners.items():
            iv = optb.binning_table.build()["IV"].sum()
            iv_data.append({
                "feature": col,
                "iv": iv
            })
        return (
            pd.DataFrame(iv_data)
            .sort_values("iv", ascending=False)
            .reset_index(drop=True)
        )

    def get_binning_table(self, col):
        return self.binners[col].binning_table.build()


class LGB_XGB_CAT_Model:

    def __init__(self, lgb_params=None, xgb_params=None, cb_params=None):

        self.lgb_model = None
        self.xgb_model = None
        self.cb_model = None

        self.lgb_params = lgb_params
        self.xgb_params = xgb_params
        self.cb_params = cb_params

    def train(self, X_train, y_train):

        if type(self.lgb_params) != type(None):
            print("Training LGB model...")
            self.lgb_model = lgb.LGBMClassifier(
                **self.lgb_params
            ).fit(
                X_train,
                y_train
            )

        if type(self.xgb_params) != type(None):
            print("Training XGB model...")
            self.xgb_model = xgb.XGBClassifier(
                **self.xgb_params
            ).fit(
                X_train,
                y_train
            )

        if type(self.cb_params) != type(None):
            print("Training CatBoost model...")
            self.cb_model = cb.CatBoostClassifier(
                **self.cb_params
            ).fit(
                X_train,
                y_train
            )
        print("Models trained successfully!")

    def predict_proba(self, X):
        probs = {}
        if type(self.lgb_params) != type(None):
            probs['lgb'] = (
                self.lgb_model
                .predict_proba(X)[:, 1]
            )

        if type(self.xgb_params) != type(None):
            probs['xgb'] = (
                self.xgb_model
                .predict_proba(X)[:, 1]
            )

        if type(self.cb_params) != type(None):
            probs['cb'] = (
                self.cb_model
                .predict_proba(X)[:, 1]
            )
        return probs

    def roc_auc_score(self, X_test, y_test):
        roc_auc_dict = {}
        if type(self.lgb_params) != type(None):
            roc_auc_dict['lgb'] = roc_auc_score(y_test, self.lgb_model.predict_proba(X_test)[:, 1])

        if type(self.xgb_params) != type(None):
            roc_auc_dict['xgb'] = roc_auc_score(y_test, self.xgb_model.predict_proba(X_test)[:, 1])

        if type(self.cb_params) != type(None):
            roc_auc_dict['cb'] = roc_auc_score(y_test, self.cb_model.predict_proba(X_test)[:, 1])

        return roc_auc_dict

    def cross_validation(self, X, y, groups, n_splits=5):
        gkf = GroupKFold(
            n_splits=n_splits
        )
        cv_scores = {
            "lgb": [],
            "xgb": [],
            "cb": []
        }

        for train_idx, valid_idx in gkf.split(X, y, groups=groups):
            X_train_cv = X.iloc[train_idx]
            X_valid_cv = X.iloc[valid_idx]

            y_train_cv = y.iloc[train_idx]
            y_valid_cv = y.iloc[valid_idx]

            if type(self.lgb_params) != type(None):

                model_lgb = lgb.LGBMClassifier(**self.lgb_params)
                model_lgb.fit(X_train_cv, y_train_cv)
                y_prob_lgb = model_lgb.predict_proba(X_valid_cv)[:, 1]
                auc_lgb = roc_auc_score(y_valid_cv, y_prob_lgb)
                cv_scores["lgb"].append(auc_lgb)

            if type(self.xgb_params) != type(None):
                model_xgb = xgb.XGBClassifier(**self.xgb_params)
                model_xgb.fit(X_train_cv, y_train_cv)
                y_prob_xgb = model_xgb.predict_proba(X_valid_cv)[:, 1]
                auc_xgb = roc_auc_score(y_valid_cv, y_prob_xgb)
                cv_scores["xgb"].append(auc_xgb)

            if type(self.cb_params) != type(None):
                model_cb = cb.CatBoostClassifier(**self.cb_params)
                model_cb.fit(X_train_cv, y_train_cv, verbose=False)
                y_prob_cb = model_cb.predict_proba(X_valid_cv)[:, 1]
                auc_cb = roc_auc_score(y_valid_cv, y_prob_cb)
                cv_scores["cb"].append(auc_cb)

        final_scores = {}

        if len(cv_scores["lgb"]) > 0:
            final_scores["lgb"] = np.mean(cv_scores["lgb"])

        if len(cv_scores["xgb"]) > 0:
            final_scores["xgb"] = np.mean(cv_scores["xgb"])

        if len(cv_scores["cb"]) > 0:
            final_scores["cb"] = np.mean(cv_scores["cb"])

        return final_scores


X, y = train_df_scaled_v1.drop(columns=[t_col, ID], axis='columns'), train_df_scaled_v1[t_col]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42,  stratify=y)
X_oot = test_df_scaled_v1.drop(columns=[ID], axis='columns').copy()

MAX_BIN, MIN_BIN = 25, 0.06

if MAX_BIN == None or MIN_BIN == None:
    max_n_bins = [5, 10, 15, 20, 25]
    min_bin_size = [0.001, 0.05, 0.06, 0.07]
    best_bin_range = {}
    best_roc_auc = 0.0

    print(f"Searching for best max_bins and min_bins combination...")
    for max_bin in max_n_bins:
        for min_bin in min_bin_size:
            print(f"Fitting with max_bin = {max_bin} and min_bin = {min_bin}...")
            bin_obj = OptimalBinner(
                c_cols=c_cols,
                n_cols=n_cols,
                max_n_bins=max_bin,
                min_bin_size=min_bin
            )

            bin_obj.fit(X_train, y_train, verbos=False)
            X_test_woe = bin_obj.transform(X_test, metric="woe")

            X_test_train, X_test_test, y_test_train, y_test_test = train_test_split(X_test_woe, y_test, test_size=0.3, random_state=42,  stratify=y_test)
            
            model_base = LGB_XGB_CAT_Model(xgb_params={})
            model_base.train(X_test_train, y_test_train)
            
            auc = model_base.roc_auc_score(X_test_test, y_test_test).values()
            avg_auc = sum(list(auc)) / len(auc)
            
            if best_roc_auc < avg_auc:
                best_roc_auc = avg_auc
                best_bin_range['max_bin'] = max_bin
                best_bin_range['min_bin'] = min_bin
            print(f"BIN RANGE : {max_bin, min_bin} | BEST BIN RANGE : {best_bin_range} | AUC : {avg_auc} | BEST AUC : {best_roc_auc}")

    MAX_BIN = best_bin_range['max_bin']
    MIN_BIN = best_bin_range['min_bin']

bin_obj = OptimalBinner(
    c_cols=c_cols,
    n_cols=n_cols,
    max_n_bins=MAX_BIN,
    min_bin_size=MIN_BIN
)

bin_obj.fit(X_train, y_train, verbos=False)

X_train_woe = bin_obj.transform(X_train, metric="woe")
X_test_woe = bin_obj.transform(X_test, metric="woe")
X_oot_woe = bin_obj.transform(X_oot, metric="woe")

print("X_train_woe shape:", X_train_woe.shape)
print("X_test_woe shape:", X_test_woe.shape)
print("X_oot_woe shape:", X_oot_woe.shape)

X_train_woe shape: (307398, 95)
X_test_woe shape: (131742, 95)
X_oot_woe shape: (188165, 95)


In [94]:
X_train_full = pd.concat([X_train_woe.rename(columns=lambda x: x + '_woe'), X_train], axis='columns')
X_test_full = pd.concat([X_test_woe.rename(columns=lambda x: x + '_woe'), X_test], axis='columns')
X_oot_full = pd.concat([X_oot_woe.rename(columns=lambda x: x + '_woe'), X_oot], axis='columns')

X_train_full.shape, X_test_full.shape, X_oot_full.shape

((307398, 190), (131742, 190), (188165, 190))

In [95]:
def submission(model, test_data, file_name:str):
    pred_data = test_data
    predictions = model.predict_proba(pred_data)[:, 1]
    sub_df['PitNextLap'] = predictions
    sub_df[['id', 'PitNextLap']].to_csv(file_name, index=False)
    print(f"Submissions saved to {file_name} path!!!!!!!!!!!!!!")

In [ ]:
count_neg = (y == 0).sum()
count_pos = (y == 1).sum()
scale_weight = count_neg / count_pos

best_xgb_params = {'max_depth': 8, 'learning_rate': 0.04101781341683126, 'min_child_weight': 80, 'subsample': 0.9440420600689602, 'colsample_bytree': 0.5538332346301195, 'colsample_bylevel': 0.8884695922566868, 'reg_alpha': 0.024826827681311267, 'reg_lambda': 0.01362810712606467, 'gamma': 0.9474271992777541}

xgb_model = XGBClassifier(
    **best_xgb_params,
    n_estimators      = 2000,
    random_state      = 42,
    n_jobs            = -1,
    eval_metric       = 'auc',
    scale_pos_weight = scale_weight
)

xgb_model.fit(X_train_full.values, y_train.values)

# --- XGBoost Evaluation ---
y_prob_test_xgb = xgb_model.predict_proba(X_test_full.values)[:, 1]
y_prob_train_xgb = xgb_model.predict_proba(X_train_full.values)[:, 1]

print("ROC AUC Score - XGBoost")
print(f"Testing:  {roc_auc_score(y_test, y_prob_test_xgb):.4f}")
print(f"Training: {roc_auc_score(y_train, y_prob_train_xgb):.4f}")

ROC AUC Score - XGBoost
Testing:  0.9510
Training: 0.9775


In [58]:

lgb_best_params = {'num_leaves': 153, 'max_depth': 8, 'learning_rate': 0.022283855802695055, 'min_child_samples': 93, 'subsample': 0.9779576595270691, 'colsample_bytree': 0.5726559097383321, 'reg_alpha': 4.979689286697664, 'reg_lambda': 0.00921590641601282}
best_params_lgb = {
    "objective"        : "binary",
    "metric"           : "auc",
    "verbosity"        : -1,
    "boosting_type"    : "gbdt",
    "scale_pos_weight" : scale_weight,
    "n_estimators"     : 2000,
    "random_state"     : 42,
    "n_jobs"           : -1,
    **lgb_best_params
}

lgb_model = LGBMClassifier(
   **best_params_lgb
)

lgb_model.fit(X_train_woe.values, y_train.values)

# --- LightGBM Evaluation ---
y_prob_test_lgb = lgb_model.predict_proba(X_test_woe.values)[:, 1]
y_prob_train_lgb = lgb_model.predict_proba(X_train_woe.values)[:, 1]

print("ROC AUC Score - LightGBM")
print(f"Testing:  {roc_auc_score(y_test, y_prob_test_lgb):.4f}")
print(f"Training: {roc_auc_score(y_train, y_prob_train_lgb):.4f}")

ROC AUC Score - LightGBM
Testing:  0.9494
Training: 0.9755


In [82]:
CONFIG = {
    # --- Model architecture ---
    "n_ens": 16,                 # renamed from n_ens
    "embedding_size": 6,
    "max_one_hot_cat_size": 4,        # renamed from onehot_thresh
    "hidden_sizes": [256, 256, 256],  # renamed from hidden_dims
    "p_drop": 0.05,                   # renamed from dropout
    "p_drop_sched": "expm4t",
    "act":'silu',                   # renamed from activation
    "add_front_scale": True,

    # --- PBLD / PLR embeddings ---
    "plr_hidden_1": 20,               # renamed from pbld_hidden_dim
    "plr_hidden_2": 4,                # pbld_out_dim - 1
    "plr_sigma": 5.0,                 # renamed from pbld_freq_scale
    "plr_act_name": 'prelu',              # renamed from pbld_activation
    "plr_lr_factor": 0.093,           # renamed from pbld_lr_factor

    # --- Optimizer ---
    "lr": 0.008,
    "mom": 0.9,
    "sq_mom": 0.98,
    "lr_sched": "flat_cos",
    "first_layer_lr_factor": 1.0,
    "scale_lr_factor": 10.0,          # renamed from lr_scale_mult
    "bias_lr_factor": 0.1,            # renamed from lr_bias_mult
    "wd": 0.005,                      # renamed from weight_decay
    "bias_wd_factor": 0.5,            # renamed from wd_bias_mult

    # --- Label smoothing ---
    "ls_eps": 0.04,
    "ls_eps_sched": "cos",

    # --- Preprocessing ---
    "tfms": ["median_center", "robust_scale", "smooth_clip"],

    # --- Training loop ---
    "n_epochs": 4,                    # renamed from epochs
    "batch_size": 256,                # renamed from train_bs
    "predict_batch_size": 10240,      # renamed from eval_bs
    "verbosity": 2,

    # --- Early stopping ---
    "use_early_stopping": False,
    "early_stopping_additive_patience": 10,
    "early_stopping_multiplicative_patience": 1,

    # --- Device ---
    "device": "cpu",
    "random_state": 42,
}

# --- Fold split ---
FOLDS = 5
SEED = 42
TE = True

In [ ]:
# The RealMLP_TD_Classifier and fitting process is unchanged
from pytabkit import RealMLP_TD_Classifier
rmlp_model = RealMLP_TD_Classifier(**CONFIG)
rmlp_model.fit(
    X_train_full.values, y_train.values,
    X_test_full.values, y_test.values,
    cat_col_names=c_cols,
)

# --- RealMLP Evaluation ---
y_prob_test_rmlp = rmlp_model.predict_proba(X_test_full.values)[:, 1]
y_prob_train_rmlp = rmlp_model.predict_proba(X_train_full.values)[:, 1]

print("ROC AUC Score - Real_MLP")
print(f"Testing:  {roc_auc_score(y_test, y_prob_test_rmlp):.4f}")
print(f"Training: {roc_auc_score(y_train, y_prob_train_rmlp):.4f}")

Columns classified as continuous: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 180, 181, 182, 183, 184, 185, 186, 187, 188, 189]
Columns classified as categorical: []


GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


Epoch 1/4: val class_error = 0.105646
Epoch 2/4: val class_error = 0.102109
Epoch 3/4: val class_error = 0.099270
Epoch 4/4: val class_error = 0.098746


`Trainer.fit` stopped: `max_epochs=4` reached.
GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


ROC AUC Score - XGBoost
Testing:  0.9517
Training: 0.9587


In [96]:
submission(rmlp_model, X_oot_full.values, "Real_MLP_Model_24_05_2026.csv")

GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


Submissions saved to Real_MLP_Model_24_05_2026.csv path!!!!!!!!!!!!!!


In [97]:
X_train_full.shape, X_oot_full.shape

((307398, 190), (188165, 190))